<a href="https://colab.research.google.com/github/Raquelodaka/LTV-e-Propensao-de-Compra/blob/main/Modelo_de_LTV_(Lifetime_Value)_e_Propens%C3%A3o_de_Compra_(Cross_Sell).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Modelo de LTV (Lifetime Value) e Propensão de Compra (Cross-Sell)
Identificar o momento ideal para oferecer novos produtos financeiros (empréstimos, seguros, investimentos) para a base atual de clientes.

Complexidade Pleno: Modelagem de propensão usando histórico de transações e saldo ao longo do tempo (séries temporais e atributos agregados).

Habilidades Técnicas:

Agregação de dados transacionais por janelas móveis (ex: gastos nos últimos 30, 60, 90 dias).

Modelagem Multirrótulo (Multi-label Classification) ou modelos independentes por produto.

Filtro colaborativo ou sistemas de recomendação híbridos.

Diferencial de Negócio: Estimar o aumento de receita anual (uplift) gerado pela conversão da oferta personalizada.


# Projeto: Next Best Action (NBA) — Propensão de Cross-Sell e Otimização de LTV

## 1. Contexto de Negócio
No setor financeiro, oferecer o produto certo para o cliente certo e no momento ideal é o principal motor de crescimento da receita por cliente. Campanhas genéricas geram alto custo operacional de contato (SMS, chamadas, push) e sobrecarregam a base com ofertas irrelevantes, diminuindo a taxa de conversão.

## 2. Objetivo do Modelo
Desenvolver uma solução analítica end-to-end capaz de:
1. **Identificar o Momento Ideal (Timing):** Mapear alterações no comportamento transacional do cliente (saldo, variação de renda e gastos) para detectar a necessidade imediata de novos produtos.
2. **Recomendar o Produto Correto (Cross-Sell):** Estimar a probabilidade individual de aquisição para Investimentos, Seguros e Empréstimos sem incorrer em vazamento de dados (*Data Leakage*).
3. **Maximizar o Retorno Financeiro (LTV Uplift):** Priorizar e ordenar as ofertas com base na **Receita Esperada**, calculada como:

$$\text{Receita Esperada} = P(\text{Conversão}) \times \text{LTV do Produto} - \text{Custo da Oferta}$$

## 3. Arquitetura da Solução
* **Estrutura dos Dados:** Dados longitudinais em painel (1.000 clientes ao longo de 6 meses).
* **Engenharia de Features:** Agregações por janelas móveis temporais (*rolling 3m*) e verificação de alteração de posse (*lag features*).
* **Modelagem:** Estratégia de aprendizado supervisionado via modelos independentes (*Binary Relevance*) com algoritmo `HistGradientBoostingClassifier`.
* **Avaliação de Negócio:** Mensuração de impacto por meio das curvas de *Cumulative Gain* e estimativa de *Uplift* financeiro líquido frente a abordagens aleatórias.

### Dataset

Registros: 6.000 linhas (1.000 clientes acompanhados ao longo de 6 meses).

Colunas:
- fecha_dato: Data do snapshot mensal (YYYY-MM-DD).
- id_cliente: Identificador único do cliente.
- idade, renda, score_credito: Atributos demográficos/cadastrais.
- saldo_mes, gastos_cartao: Comportamento financeiro mensal.
- ind_investimento, ind_seguro, ind_emprestimo: Indicadores de posse do produto no mês (0 ou 1).

In [ ]:
# 1. Instalar o Sweetviz
!pip install -q sweetviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 60.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

# Pré-processamento e Validação
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Algoritmos de Machine Learning
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

In [ ]:
# Carregar o arquivo enviado
df = pd.read_csv('/content/base_uso_cartao_6_meses.csv')

# Visualizar
df

,fecha_dato,id_cliente,idade,renda,score_credito,saldo_mes,gastos_cartao,ind_investimento,ind_seguro,ind_emprestimo
0,2025-01-01,10001,56,7073.05,570,7305.86,3026.59,0,0,0
1,2025-02-01,10001,56,7073.05,570,5829.62,2501.44,0,0,0
2,2025-03-01,10001,56,7073.05,570,5552.94,2526.46,0,1,0
3,2025-04-01,10001,56,7073.05,570,6406.27,2444.18,0,1,0
4,2025-05-01,10001,56,7073.05,570,6003.65,2404.39,0,1,0
...,...,...,...,...,...,...,...,...,...,...
5995,2025-02-01,11000,25,NaN,593,4292.96,824.87,1,0,0
5996,2025-03-01,11000,25,2115.17,593,6662.75,992.05,1,0,0
5997,2025-04-01,11000,25,2115.17,593,6793.83,521.93,1,0,0
5998,2025-05-01,11000,25,2115.17,593,6596.44,895.74,1,0,0


## Tratamento de Valores Ausentes (Imputação na Variável Renda)

### Justificativa Técnica e de Negócio
Em dados financeiros longitudinais (séries temporais por cliente), apagar linhas com valores nulos causaria a perda do histórico temporal do cliente, prejudicando a criação das janelas móveis temporais (*lags*).

Para resolver os valores ausentes na coluna `renda`, adotamos uma estratégia em duas camadas:

1. **Imputação Temporal por Cliente (`ffill` + `bfill`):**
   * **Propagação para Frente (`ffill` / Forward Fill):** Preenche o valor ausente do mês atual com a renda declarada pelo próprio cliente no mês anterior, assumindo estabilidade financeira no curto prazo.
   * **Propagação para Trás (`bfill` / Backward Fill):** Caso o valor nulo esteja no primeiro mês do histórico, o código busca a primeira renda registrada nos meses seguintes do próprio cliente.

2. **Imputação pela Mediana Geral (`fillna(median)`):**
   * Caso existam clientes que **nunca** registraram a renda em nenhum dos 6 meses, os valores restantes são preenchidos com a **mediana global da renda**.
   * *Por que a mediana e não a média?* A renda costuma apresentar uma distribuição assimétrica (com cauda longa à direita devido a rendas muito altas). A mediana é uma medida robusta que não sofre distorção por *outliers*.

In [ ]:
#tratamento de nulos na renda mantendo histórico do cliente
df['renda'] = df.groupby('id_cliente')['renda'].transform(lambda x: x.ffill().bfill())
df['renda'] = df['renda'].fillna(df['renda'].median())
df

,fecha_dato,id_cliente,idade,renda,score_credito,saldo_mes,gastos_cartao,ind_investimento,ind_seguro,ind_emprestimo
0,2025-01-01,10001,56,7073.05,570,7305.86,3026.59,0,0,0
1,2025-02-01,10001,56,7073.05,570,5829.62,2501.44,0,0,0
2,2025-03-01,10001,56,7073.05,570,5552.94,2526.46,0,1,0
3,2025-04-01,10001,56,7073.05,570,6406.27,2444.18,0,1,0
4,2025-05-01,10001,56,7073.05,570,6003.65,2404.39,0,1,0
...,...,...,...,...,...,...,...,...,...,...
5995,2025-02-01,11000,25,2115.17,593,4292.96,824.87,1,0,0
5996,2025-03-01,11000,25,2115.17,593,6662.75,992.05,1,0,0
5997,2025-04-01,11000,25,2115.17,593,6793.83,521.93,1,0,0
5998,2025-05-01,11000,25,2115.17,593,6596.44,895.74,1,0,0


## Tratamento de Outliers (Winsorização / Capping na Variável Saldo)

### Justificativa Técnica e de Negócio
Em bases de dados de instituições financeiras, a variável `saldo_mes` costuma apresentar **valores extremamente altos (outliers)** pertencentes a um número reduzido de clientes de altíssima renda ou oriundos de movimentações atípicas pontuais.

Em vez de remover essas linhas — o que destruiria o histórico longitudinal do cliente e reduziria a base —, aplicamos a técnica de **Winsorização (Capping)** no **Percentil 99 ($P_{99}$)**.

### Como a Técnica Funciona
1. **Identificação do Teto ($P_{99}$):** O código calcula o limite máximo onde se encontram 99% dos menores saldos da base.
2. **Limitador Superior (`np.where`):** Qualquer valor de saldo que supere o $P_{99}$ é substituído pelo próprio valor do $P_{99}$. Saldos menores ou iguais permanecem inalterados.

### Impacto no Modelo
* **Proteção contra Distorção:** Evita que coeficientes do modelo e divisões de nós nas árvores de decisão sejam dominados por valores discrepantes.
* **Preservação de Dados:** Retém 100% dos registros e clientes na base, mantendo a ordem de grandeza do saldo (um saldo alto continua sendo classificado como alto, mas dentro de uma escala controlada).

In [ ]:
#Tratamento de Outliers (Capping no Percentil 99 para saldo_mes)
q99 = df['saldo_mes'].quantile(0.99)
df['saldo_mes'] = np.where(df['saldo_mes'] > q99, q99, df['saldo_mes'])
df

,fecha_dato,id_cliente,idade,renda,score_credito,saldo_mes,gastos_cartao,ind_investimento,ind_seguro,ind_emprestimo
0,2025-01-01,10001,56,7073.05,570,7305.86,3026.59,0,0,0
1,2025-02-01,10001,56,7073.05,570,5829.62,2501.44,0,0,0
2,2025-03-01,10001,56,7073.05,570,5552.94,2526.46,0,1,0
3,2025-04-01,10001,56,7073.05,570,6406.27,2444.18,0,1,0
4,2025-05-01,10001,56,7073.05,570,6003.65,2404.39,0,1,0
...,...,...,...,...,...,...,...,...,...,...
5995,2025-02-01,11000,25,2115.17,593,4292.96,824.87,1,0,0
5996,2025-03-01,11000,25,2115.17,593,6662.75,992.05,1,0,0
5997,2025-04-01,11000,25,2115.17,593,6793.83,521.93,1,0,0
5998,2025-05-01,11000,25,2115.17,593,6596.44,895.74,1,0,0


##Feature Engineering: Ordenação Temporal e Conversão de Datas

### Justificativa Técnica e de Negócio
Para modelar dados longitudinais (painel temporal) sem introduzir **vazamento de dados (*Data Leakage*)**, é fundamental garantir que os registros estejam perfeitamente estruturados e ordenados no tempo.

Nesta etapa, preparamos a base para a criação de variáveis defasadas (*lags*) e janelas móveis (*rolling features*).

### Como a Técnica Funciona
1. **Conversão de Tipo (`pd.to_datetime`):** Transforma a coluna `fecha_dato` de formato texto (string) para o tipo nativo de data do Pandas (`datetime64`). Isso permite realizar operações cronológicas e ordenações corretas.
2. **Ordenação Temporal do Cliente (`sort_values`):** Organiza a base primeiro por `id_cliente` e, em seguida, por `fecha_dato` em ordem crescente (do mês mais antigo para o mais recente).
3. **Redefinição de Índice (`reset_index(drop=True)`):** Recria os índices numéricos das linhas de forma contínua após a reordenação, evitando inconsistências em operações futuras.

### Por que esta etapa é crítica?
A criação das variáveis de *lag* (como `.shift(1)`) e de janelas móveis dependem estritamente do elemento anterior na sequência. Se a base não estiver ordenada por data para cada cliente, o modelo usará dados do futuro para prever o passado ou misturará dados de clientes diferentes.

In [ ]:
#FEATURE ENGINEERING & TEMPORAL LAGS
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])
df = df.sort_values(['id_cliente', 'fecha_dato']).reset_index(drop=True)
df

,fecha_dato,id_cliente,idade,renda,score_credito,saldo_mes,gastos_cartao,ind_investimento,ind_seguro,ind_emprestimo
0,2025-01-01,10001,56,7073.05,570,7305.86,3026.59,0,0,0
1,2025-02-01,10001,56,7073.05,570,5829.62,2501.44,0,0,0
2,2025-03-01,10001,56,7073.05,570,5552.94,2526.46,0,1,0
3,2025-04-01,10001,56,7073.05,570,6406.27,2444.18,0,1,0
4,2025-05-01,10001,56,7073.05,570,6003.65,2404.39,0,1,0
...,...,...,...,...,...,...,...,...,...,...
5995,2025-02-01,11000,25,2115.17,593,4292.96,824.87,1,0,0
5996,2025-03-01,11000,25,2115.17,593,6662.75,992.05,1,0,0
5997,2025-04-01,11000,25,2115.17,593,6793.83,521.93,1,0,0
5998,2025-05-01,11000,25,2115.17,593,6596.44,895.74,1,0,0


## Criação da Variável Alvo (Target de Aquisição / Cross-Sell)

### Justificativa Técnica e de Negócio
Para prever a **propensão de contratação**, não basta identificar se o cliente possui o produto no mês atual. Precisamos identificar o **momento exato da conversão (aquisição)**, ou seja, a transição em que o cliente **não possuía** o produto no mês anterior ($T-1$) e **passou a possuir** no mês atual ($T$).

Esta abordagem garante a criação de um modelo supervisionado sem vazamento de dados (*Data Leakage*).

### Como a Técnica Funciona
Para cada um dos produtos (`ind_investimento`, `ind_seguro`, `ind_emprestimo`):

1. **Criação da Defasagem Temporal (`shift(1)`):**
   * O método `df.groupby('id_cliente')[p].shift(1)` desloca o status do produto em 1 mês para baixo.
   * Cria a coluna `lag_produto`, que guarda o estado da posse do produto no mês $T-1$.

2. **Regra de Conversão (Target Binário):**
   * Avalia a condição booleana: `(lag_produto == 0) & (produto == 1)`.
   * **`target = 1`:** Indica uma **nova contratação** feita no mês atual (sucesso de Cross-Sell).
   * **`target = 0`:** Indica que o cliente já possuía o produto ou continuou sem contratá-lo.

In [ ]:
#Variável Alvo: Cliente não tinha no mês T-1 e adquiriu no mês T
produtos = ['ind_investimento', 'ind_seguro', 'ind_emprestimo']
for p in produtos:
    df[f'lag_{p}'] = df.groupby('id_cliente')[p].shift(1)
    df[f'target_{p}'] = ((df[f'lag_{p}'] == 0) & (df[p] == 1)).astype(int)
df

,fecha_dato,id_cliente,idade,renda,score_credito,saldo_mes,gastos_cartao,ind_investimento,ind_seguro,ind_emprestimo,lag_ind_investimento,target_ind_investimento,lag_ind_seguro,target_ind_seguro,lag_ind_emprestimo,target_ind_emprestimo
0,2025-01-01,10001,56,7073.05,570,7305.86,3026.59,0,0,0,NaN,0,NaN,0,NaN,0
1,2025-02-01,10001,56,7073.05,570,5829.62,2501.44,0,0,0,0.0,0,0.0,0,0.0,0
2,2025-03-01,10001,56,7073.05,570,5552.94,2526.46,0,1,0,0.0,0,0.0,1,0.0,0
3,2025-04-01,10001,56,7073.05,570,6406.27,2444.18,0,1,0,0.0,0,1.0,0,0.0,0
4,2025-05-01,10001,56,7073.05,570,6003.65,2404.39,0,1,0,0.0,0,1.0,0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,2025-02-01,11000,25,2115.17,593,4292.96,824.87,1,0,0,1.0,0,0.0,0,0.0,0
5996,2025-03-01,11000,25,2115.17,593,6662.75,992.05,1,0,0,1.0,0,0.0,0,0.0,0
5997,2025-04-01,11000,25,2115.17,593,6793.83,521.93,1,0,0,1.0,0,0.0,0,0.0,0
5998,2025-05-01,11000,25,2115.17,593,6596.44,895.74,1,0,0,1.0,0,0.0,0,0.0,0


## 6. Agregações Móveis e Indicadores de Saúde Financeira (Rolling Features)

### Justificativa Técnica e de Negócio
Variáveis estáticas de um único mês (fotos pontuais) podem sofrer oscilações atípicas por conta de sazonalidades. Para capturar a **tendência de comportamento** do cliente ao longo do tempo e criar gatilhos (*triggers*) mais consistentes de *Cross-Sell*, criamos variáveis agregadas com janelas móveis de **3 meses** (`rolling(3)`).

Essas variáveis mostram a real capacidade financeira e o nível de consumo acumulado pelo cliente.

### Como a Técnica Funciona
1. **Médias Móveis de 3 Meses (`rolling(3).mean()`):**
   * **`media_saldo_3m`:** Calcula a média do saldo mantido pelo cliente nos últimos 3 meses. Clientes com média de saldo alta e crescente são fortes candidatos para ofertas de **Investimento**.
   * **`media_gastos_3m`:** Avalia o patamar contínuo de consumo do cliente no cartão de crédito.
   * **`min_periods=1`:** Garante que mesmo nos primeiros meses do histórico (quando ainda não há 3 meses completos acumulados), a média seja calculada com os meses disponíveis (evitando a criação de valores nulos `NaN`).

2. **Razão do Endividamento / Comprometimento (`ratio_gastos_renda`):**
   * **`gastos_cartao / (renda + 1)`:** Mede a proporção da renda comprometida com cartão de crédito no mês.
   * **Sinalizador de Propensão:** Clientes com *ratio* elevado (gastos próximos ou superiores à renda) têm alta propensão a necessitar de **Empréstimos** para reorganização financeira.
   * **`+ 1` no denominador:** Aplica um pequeno ajuste de estabilidade numérica (evita erro de divisão por zero caso ocorra renda zerada).

In [ ]:
#Agregações Móveis (Janela de 3 meses)
df['media_saldo_3m'] = df.groupby('id_cliente')['saldo_mes'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['media_gastos_3m'] = df.groupby('id_cliente')['gastos_cartao'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['ratio_gastos_renda'] = df['gastos_cartao'] / (df['renda'] + 1)
df

,fecha_dato,id_cliente,idade,renda,score_credito,saldo_mes,gastos_cartao,ind_investimento,ind_seguro,ind_emprestimo,lag_ind_investimento,target_ind_investimento,lag_ind_seguro,target_ind_seguro,lag_ind_emprestimo,target_ind_emprestimo,media_saldo_3m,media_gastos_3m,ratio_gastos_renda
0,2025-01-01,10001,56,7073.05,570,7305.86,3026.59,0,0,0,NaN,0,NaN,0,NaN,0,7305.860000,3026.590000,0.427844
1,2025-02-01,10001,56,7073.05,570,5829.62,2501.44,0,0,0,0.0,0,0.0,0,0.0,0,6567.740000,2764.015000,0.353608
2,2025-03-01,10001,56,7073.05,570,5552.94,2526.46,0,1,0,0.0,0,0.0,1,0.0,0,6229.473333,2684.830000,0.357145
3,2025-04-01,10001,56,7073.05,570,6406.27,2444.18,0,1,0,0.0,0,1.0,0,0.0,0,5929.610000,2490.693333,0.345514
4,2025-05-01,10001,56,7073.05,570,6003.65,2404.39,0,1,0,0.0,0,1.0,0,0.0,0,5987.620000,2458.343333,0.339889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,2025-02-01,11000,25,2115.17,593,4292.96,824.87,1,0,0,1.0,0,0.0,0,0.0,0,5155.030000,925.595000,0.389794
5996,2025-03-01,11000,25,2115.17,593,6662.75,992.05,1,0,0,1.0,0,0.0,0,0.0,0,5657.603333,947.746667,0.468795
5997,2025-04-01,11000,25,2115.17,593,6793.83,521.93,1,0,0,1.0,0,0.0,0,0.0,0,5916.513333,779.616667,0.246639
5998,2025-05-01,11000,25,2115.17,593,6596.44,895.74,1,0,0,1.0,0,0.0,0,0.0,0,6684.340000,803.240000,0.423284


## Descarte do Mês Inicial (Sanitização do Dataset de Treino)

### Justificativa Técnica e de Negócio
No primeiro mês do nosso histórico temporal ($T=1$), não existe um mês anterior ($T=0$) registrado no dataset. Por isso, as colunas de defasagem (`lag_produto`) resultam em valores ausentes (`NaN`).

Como a definição da nossa **variável alvo (target)** depende estritamente de comparar a posse do produto entre o mês anterior e o atual, o primeiro mês não possui informação suficiente para determinar se houve ou não uma nova aquisição.

### Como a Técnica Funciona
* **`df.dropna(subset=['lag_ind_emprestimo'])`**: Remove da base de modelagem apenas as linhas em que a coluna `lag_ind_emprestimo` possui valor nulo (`NaN`). Como o cálculo de *lag* foi feito de forma alinhada para todos os produtos, essa operação elimina exatamente o mês inicial ($T=1$) de todos os clientes.
* **`.copy()`**: Cria uma cópia independente do DataFrame na memória (`df_model`), evitando avisos de modificação em visualizações (*SettingWithCopyWarning*) nas etapas posteriores de modelagem.

### Impacto na Qualidade do Modelo
Esta limpeza garante que o modelo de Machine Learning seja treinado **exclusivamente com dados válidos e completos**, eliminando ambiguidades e mantendo a integridade matemática das taxas de conversão.

In [ ]:
#Descartar mês inicial (sem histórico prévio para verificar alteração de posse)
df_model = df.dropna(subset=['lag_ind_emprestimo']).copy()
df_model

,fecha_dato,id_cliente,idade,renda,score_credito,saldo_mes,gastos_cartao,ind_investimento,ind_seguro,ind_emprestimo,lag_ind_investimento,target_ind_investimento,lag_ind_seguro,target_ind_seguro,lag_ind_emprestimo,target_ind_emprestimo,media_saldo_3m,media_gastos_3m,ratio_gastos_renda
1,2025-02-01,10001,56,7073.05,570,5829.62,2501.44,0,0,0,0.0,0,0.0,0,0.0,0,6567.740000,2764.015000,0.353608
2,2025-03-01,10001,56,7073.05,570,5552.94,2526.46,0,1,0,0.0,0,0.0,1,0.0,0,6229.473333,2684.830000,0.357145
3,2025-04-01,10001,56,7073.05,570,6406.27,2444.18,0,1,0,0.0,0,1.0,0,0.0,0,5929.610000,2490.693333,0.345514
4,2025-05-01,10001,56,7073.05,570,6003.65,2404.39,0,1,0,0.0,0,1.0,0,0.0,0,5987.620000,2458.343333,0.339889
5,2025-06-01,10001,56,7073.05,570,5074.50,3411.01,0,1,0,0.0,0,1.0,0,0.0,0,5828.140000,2753.193333,0.482186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,2025-02-01,11000,25,2115.17,593,4292.96,824.87,1,0,0,1.0,0,0.0,0,0.0,0,5155.030000,925.595000,0.389794
5996,2025-03-01,11000,25,2115.17,593,6662.75,992.05,1,0,0,1.0,0,0.0,0,0.0,0,5657.603333,947.746667,0.468795
5997,2025-04-01,11000,25,2115.17,593,6793.83,521.93,1,0,0,1.0,0,0.0,0,0.0,0,5916.513333,779.616667,0.246639
5998,2025-05-01,11000,25,2115.17,593,6596.44,895.74,1,0,0,1.0,0,0.0,0,0.0,0,6684.340000,803.240000,0.423284


## Seleção de Elegibilidade, Separação do Dataset e Treinamento do Modelo

### Justificativa Técnica e de Negócio
Nem todo cliente da base deve receber uma oferta de empréstimo. Oferecer um produto para quem **já o possui contratado** gera custos operacionais indevidos e insatisfação da base.

Além disso, esta etapa separa rigorosamente as variáveis explicativas ($X$) da variável alvo ($y$) e divide os dados para validação do algoritmo.

### Como a Técnica Funciona

1. **Filtro de Elegibilidade (`lag_ind_emprestimo == 0`):**
   * Selecionamos apenas as observações de clientes que **não possuíam empréstimo no mês anterior**.
   * Garante que o modelo aprenda a prever a **primeira contratação/conversão** e previne viés na taxa de resposta.

2. **Definição da Matriz de Features ($X$) e Target ($y$):**
   * **`features` ($X$):** Conjunto de variáveis explicativas cobrindo perfil demográfico (`idade`, `score_credito`), capacidade financeira (`renda`, `saldo_mes`, `gastos_cartao`) e histórico recente (`media_saldo_3m`, `media_gastos_3m`, `ratio_gastos_renda`).
   * **Target ($y$):** Variável binária (`1` para quem contratou o empréstimo no mês e `0` para quem continuou sem contratar).

3. **Divisão Treino/Teste com Estratificação (`train_test_split`):**
   * **`test_size=0.25`:** Reserva 25% dos dados exclusivamente para testar a capacidade de generalização do modelo em dados não vistos.
   * **`stratify=y`:** Preserva a mesma proporção exata da classe rara (conversões de empréstimo) tanto no conjunto de treino quanto no de teste, evitando amostragem tendenciosa.

4. **Tratamento de Desbalanceamento (`class_weight='balanced'`):**
   * Como a taxa de conversão em produtos financeiros costuma ser baixa (classe `1` muito mais rara que a `0`), a opção `class_weight='balanced'` reajusta os pesos do algoritmo inversamente proporcional à frequência das classes.
   * Isso impede que o modelo fique viciado em prever apenas a classe majoritária (`0`).

In [ ]:
#TREINAMENTO DO MODELO (PROPENSÃO A EMPRÉSTIMO)
# Seleção de clientes elegíveis (apenas quem ainda não possui o produto)
df_emp = df_model[df_model['lag_ind_emprestimo'] == 0].copy()

features = ['idade', 'renda', 'score_credito', 'saldo_mes', 'gastos_cartao',
            'media_saldo_3m', 'media_gastos_3m', 'ratio_gastos_renda']

X = df_emp[features]
y = df_emp['target_ind_emprestimo']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = HistGradientBoostingClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

HistGradientBoostingClassifier(class_weight='balanced', random_state=42)

## Avaliação Preditiva e Cálculo do Uplift Financeiro (Business Value)

### Justificativa Técnica e de Negócio
Em problemas de negócio, a métrica de acurácia tradicional do modelo não é suficiente. O objetivo final não é apenas prever quem vai converter, mas sim **otimizar a alocação de recursos operacionais** (custo por contato) para maximizar o **retorno financeiro líquido (ROI/Uplift)**.

Nesta etapa, transformamos as saídas probabilísticas do modelo em decisões financeiras acionáveis.

### Como a Técnica Funciona

1. **Extração da Probabilidade de Conversão (`predict_proba`):**
   * Em vez de usar classificações rígidas (`0` ou `1`), usamos `predict_proba(X_test)[:, 1]` para extrair a **probabilidade contínua (0% a 100%)** do cliente contratar o produto no mês.
   * Isso nos permite ordenar a base de clientes do mais propenso ao menos propenso.

2. **Definição das Premissas de Negócio:**
   * **`LTV_EMPRESTIMO` (R$ 1.500,00):** Estimativa da margem líquida / valor gerado ao longo da vida útil do cliente ao adquirir um empréstimo.
   * **`CUSTO_OFERTA` (R$ 10,00):** Custo operacional unitário para contatar o cliente (custo de canais como SMS, WhatsApp, e-mail marketing ou agentes).

3. **Estratégia de Seleção do Top K% (Foco Operacional):**
   * Em vez de contatar toda a base, focamos nos **20% clientes mais propensos** (Top Decis).
   * **Cálculo da Receita Líquida do Modelo:**
     $$\text{Receita Líquida} = (\text{Conversões Reais no Top 20\%} \times \text{LTV}) - (\text{Total de Ofertas} \times \text{Custo})$$

4. **Mensuração do Uplift Financeiro:**
   * Comparamos o resultado do modelo contra uma **abordagem aleatória (Sem Modelo)** de mesmo tamanho.
   * O **Uplift** representa a **receita adicional líquida** que o modelo entrega diretamente para o negócio ao substituir campanhas não direcionadas.

In [ ]:
#AVALIAÇÃO DE DESEMPENHO E UPLIFT FINANCEIRO
y_probs = model.predict_proba(X_test)[:, 1]

## Seleção do Top K% (Foco nos 20% Mais Propensos e Estruturação do Dataframe)

### Justificativa Técnica e de Negócio
Em campanhas de marketing e vendas, o orçamento e a capacidade de atendimento são limitados. Disparar ofertas para toda a base de clientes gera custos desnecessários e fadiga na comunicação.

A estratégia do **Top K%** utiliza as probabilidades geradas pelo modelo para **ordenar os clientes da maior para a menor propensão**, focando os esforços operacionais nos 20% clientes com maior chance de conversão.

### Como a Técnica Funciona

1. **Atribuição das Probabilidades ao Dataframe (`df_res['prob']`):**
   * Unimos os dados do conjunto de teste (`X_test`), o resultado real de conversão (`y_test`) e a probabilidade estimada pelo modelo (`y_probs`) em um único DataFrame de resultados (`df_res`).

2. **Ordenação Preditiva (`sort_values('prob', ascending=False)`):**
   * Reorganizamos todos os clientes em ordem decrescente de probabilidade de compra. Os clientes no topo do DataFrame representam o público prioritário para a abordagem comercial.

3. **Cálculo da Corte de Corte / Top K (`int(len(df_res) * 0.20)`):**
   * Determinamos o tamanho exato da amostra correspondente aos 20% mais propensos (o primeiro quintil da base).

4. **Isolamento da Campanha (`df_res.iloc[:top_k]`):**
   * Selecionamos os registros do início do DataFrame até a posição `top_k`. É sobre esse grupo restrito que calculamos o retorno financeiro da campanha e o ROI gerado pelo modelo.

In [ ]:
# Simulação da Campanha (Foco nos 20% mais propensos)
LTV_EMPRESTIMO = 1500.00
CUSTO_OFERTA = 10.00

df_res = X_test.copy()
df_res['target_real'] = y_test
df_res['prob'] = y_probs
df_res = df_res.sort_values('prob', ascending=False).reset_index(drop=True)

top_k = int(len(df_res) * 0.20)
campanha = df_res.iloc[:top_k]

## Comparação de Performance e Cálculo do Uplift Financeiro Líquido

### Justificativa Técnica e de Negócio
Para provar o valor de um modelo de Machine Learning em produção, não basta mostrar que ele funciona; é necessário provar que ele traz **mais dinheiro líquido para o negócio** do que a estratégia atual ou uma abordagem ingênua (como selecionar clientes de forma aleatória).

Nesta etapa final, comparamos os resultados de conversão e receita gerados pela seleção preditiva do modelo contra uma amostragem aleatória do mesmo tamanho ($Top K$).

### Como a Técnica Funciona

1. **Cálculo da Receita Líquida do Modelo (`receita_liquida_modelo`):**
   * **`conversoes_modelo`:** Soma das conversões reais que ocorreram dentro do grupo das pessoas recomendadas pelo modelo ($Top 20\%$).
   * **Receita Bruta do Modelo:** $\text{conversoes\_modelo} \times \text{LTV\_EMPRESTIMO}$ (R$ 1.500,00).
   * **Custo do Modelo:** $\text{top\_k} \times \text{CUSTO\_OFERTA}$ (R$ 10,00 por cliente contatado).
   * **Receita Líquida:** Receita Bruta acumulada deduzida do custo total da campanha.

2. **Simulação do Grupo de Controle / Baseline (`receita_liquida_aleatoria`):**
   * **`conversoes_aleatorias`:** Seleciona um grupo de tamanho equivalente ($\text{top\_k}$) de forma puramente aleatória da base de teste (`sample(n=top_k, random_state=42)`).
   * Calcula a Receita Líquida que o negócio teria obtido se fizesse disparos de ofertas sem o uso de Inteligência Artificial.

3. **Cálculo do Uplift Financeiro Líquido (`uplift`):**
   * $\text{Uplift} = \text{Receita Líquida do Modelo} - \text{Receita Líquida Aleatória}$
   * Representa o **valor monetário incremental direto (R$)** que o algoritmo adiciona ao resultado da empresa ao substituir uma campanha de marketing tradicional por uma campanha direcionada por propensão.

In [ ]:
# Resultados financeiros executados sobre os dados de teste
conversoes_modelo = campanha['target_real'].sum()
receita_liquida_modelo = (conversoes_modelo * LTV_EMPRESTIMO) - (top_k * CUSTO_OFERTA)

conversoes_aleatorias = df_res['target_real'].sample(n=top_k, random_state=42).sum()
receita_liquida_aleatoria = (conversoes_aleatorias * LTV_EMPRESTIMO) - (top_k * CUSTO_OFERTA)

uplift = receita_liquida_modelo - receita_liquida_aleatoria

In [ ]:
# Exibição formatada dos resultados financeiros
print("=" * 55)
print("       RELATÓRIO DE IMPACTO FINANCEIRO E UPLIFT")
print("=" * 55)
print(f"Total de Clientes Contatados (Top 20%): {top_k}")
print("-" * 55)
print(f"Conversões do Modelo:              {conversoes_modelo} clientes")
print(f"Receita Líquida do Modelo:          R$ {receita_liquida_modelo:,.2f}")
print("-" * 55)
print(f"Conversões Aleatórias (Controle):  {conversoes_aleatorias} clientes")
print(f"Receita Líquida Aleatória:          R$ {receita_liquida_aleatoria:,.2f}")
print("=" * 55)
print(f"UPLIFT FINANCEIRO GERADO:          R$ {uplift:,.2f}")
print("=" * 55)

       RELATÓRIO DE IMPACTO FINANCEIRO E UPLIFT
Total de Clientes Contatados (Top 20%): 233
-------------------------------------------------------
Conversões do Modelo:              7 clientes
Receita Líquida do Modelo:          R$ 8,170.00
-------------------------------------------------------
Conversões Aleatórias (Controle):  2 clientes
Receita Líquida Aleatória:          R$ 670.00
UPLIFT FINANCEIRO GERADO:          R$ 7,500.00
